In [ ]:
import sys
import os
import pickle
from tqdm.auto import tqdm

BASE_DIR = os.getcwd()
BASE_DIR

In [ ]:
# Set the primary font for visualization
from matplotlib import font_manager as fm

# Define the font path and initialize FontProperties
font_path = BASE_DIR + '/NotoSans-VariableFont_wdth,wght.ttf'
prop = fm.FontProperties(fname=font_path)

# Step3: Hyperlink network and editors

## Step3-1: Community detecting

### Step 3-1-1: Check hyperlinked network

In [120]:
import os
import pickle
import networkx as nx
from collections import Counter

# DATA_DIR: Source directory for conditioned interaction datasets
DATA_DIR = os.path.join(BASE_DIR, 'Dataset')

# Define the absolute file path for the validated interaction pairs
pairs_file_path = os.path.join(DATA_DIR, 'conditioned_history', 'condi_pairs_1000_1000.pickle')

with open(pairs_file_path, 'rb') as fr:
    pairs = pickle.load(fr)

# Initialize directed and undirected graph structures for topological analysis
G_di = nx.DiGraph()
G_undi = nx.Graph()

# Calculate edge weights based on the frequency of observed interactions between nodes
edge_counts = Counter(pairs)

# Populate the network graphs with weighted edges to represent link frequency
for (u, v), weight in edge_counts.items():
    # Directed edges preserve the orientation of hyperlinks between source and target pages
    G_di.add_edge(u, v, weight=weight)
    # Undirected edges represent symmetrical connectivity for structural property analysis
    G_undi.add_edge(u, v, weight=weight)

### Step 3-1-2 Leiden algorithm

In [ ]:
import os
import pickle
from collections import Counter
import networkx as nx
import igraph as ig
import leidenalg

DATA_DIR = os.path.join(BASE_DIR, 'Dataset')


def Leiden_Run(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Executes a single iteration of the Leiden community detection algorithm.
    Identifies optimal network partitions by maximizing modularity while considering edge weights.
    """
    
    # DATA_DIR: Source directory for processed datasets and network files
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset')
    
    # Construct the absolute file path for the conditioned interaction pairs
    pairs_file_path = os.path.join(DATA_DIR, 'conditioned_history', f'condi_pairs_{page_edit_number}_{editor_edit_number}.pickle')
    
    with open(pairs_file_path, 'rb') as fr:
        pairs = pickle.load(fr)

    # Initialize a directed graph to preserve asymmetric adjacency relationships
    G = nx.DiGraph()
    edge_counts = Counter(pairs)
    for (u, v), weight in edge_counts.items():
        G.add_edge(u, v, weight=weight)
        
    # Mapping identifiers to integer indices for structural compatibility with igraph
    sorted_nodes = sorted(G.nodes()) # 정렬하여 순서 고정
    nodes_map = {node: i for i, node in enumerate(sorted_nodes)}
    reverse_map = {i: node for node, i in nodes_map.items()}
    
    def nx_to_igraph(nx_graph):
        """
        Translates a NetworkX graph object into an igraph structure for high-performance clustering.
        """
        g = ig.Graph(directed=nx_graph.is_directed())
        g.add_vertices(len(sorted_nodes))
        g.vs["name"] = sorted_nodes

        # Extracting edges and weights using pre-defined integer mapping
        edge_list = [(nodes_map[u], nodes_map[v]) for u, v, data in nx_graph.edges(data=True)]
        weights = [data.get('weight', 1) for u, v, data in nx_graph.edges(data=True)]
        
        g.add_edges(edge_list)
        g.es["weight"] = weights 
        
        return g
    
    G_ig = nx_to_igraph(G)
    
    print("=== Initiating Leiden Community Detection (Single Run) ===")

    # Define standardized file paths for reporting and log serialization
    log_file_path = os.path.join(DATA_DIR, f"leiden_report_{page_edit_number}_{editor_edit_number}.txt")
    rand_seed = 20260206
    with open(log_file_path, 'w', encoding='utf-8') as f:
        def log_and_print(message):
            print(message)
            f.write(message + '\n')
        log_and_print(f"=== seed: {rand_seed} ===")
        log_and_print(f"=== Network Loaded: {G.number_of_nodes()} Nodes, {G.number_of_edges()} Edges ===")
        
        # Optimize the partition based on the ModularityVertexPartition objective function
        final_partition = leidenalg.find_partition(
            G_ig, 
            leidenalg.ModularityVertexPartition, 
            weights='weight',
            seed=rand_seed
        )
    
        # Map localized indices back to original node identifiers
        final_membership = final_partition.membership
        node_to_partition = {
            reverse_map[idx]: comm_id 
            for idx, comm_id in enumerate(final_membership)
        }
        partition_sizes = final_partition.sizes()
            
        log_and_print("=== Community Detection Complete ===")
        log_and_print(f"Final Modularity Score: {final_partition.modularity:.4f}")
        log_and_print(f"Total Communities Identified: {len(final_partition)}")

        for comm_id, size in enumerate(final_partition.sizes()):
            log_and_print(f"Community {comm_id}: {size} Nodes")
        
        # Serialize the final node-to-partition mapping
        output_file_path = os.path.join(DATA_DIR, f"leiden_partition_{page_edit_number}_{editor_edit_number}.pickle")
        with open(output_file_path, "wb") as fw:
            pickle.dump(node_to_partition, fw)

        print(f"Results successfully serialized to: {output_file_path}")
        
    return

### Step 3-1-3 community information

In [ ]:
import os
import pickle

def Page_name_per_community(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Maps localized node identifiers to their corresponding Wikipedia page titles within identified communities.
    The resulting community-wise page lists are serialized into text files for qualitative content analysis.
    """
    
    # DATA_DIR: Source directory for processed network data and metadata
    # TARGET_DIR: Destination directory for community-based analytical results
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset')
    TARGET_DIR = os.path.join(BASE_DIR, 'Results')
    
    # Construct the absolute file path for the Leiden partition results
    leiden_partition_file_path = os.path.join(DATA_DIR, f"leiden_partition_{page_edit_number}_{editor_edit_number}.pickle")
    with open(leiden_partition_file_path, "rb") as fr:
        node_to_partition = pickle.load(fr)
    
    # Construct the absolute file path for the page metadata (ID-to-Title mapping)
    page_id_file_path = os.path.join(DATA_DIR, "raw_page2page", "page_id_dict.pickle")
    with open(page_id_file_path, "rb") as fr:
        page_id_dict = pickle.load(fr)
    
    # Initialize a dictionary to aggregate page titles categorized by community indices
    page_name_commu = {loc: [] for loc in range(19)}
    
    # Translate page IDs into human-readable titles using the loaded metadata
    for page in node_to_partition.keys():
        # page_id_dict[id][0] returns the canonical title of the Wikipedia article
        page_name_commu[node_to_partition[page]].append(page_id_dict[int(page)][0])
        
        
    # Define and ensure the existence of the specific output directory for the current thresholds
    community_results_dir = os.path.join(TARGET_DIR, f"page_name_per_community_{page_edit_number}_{editor_edit_number}")
    os.makedirs(community_results_dir, exist_ok=True)
    # Export community-specific page titles into individual text files
    for loc in range(19):
        output_file_path = os.path.join(community_results_dir, f"page_name_{loc}.txt")

        with open(output_file_path, "w", encoding="utf-8") as file:
            file.write(f"Community {loc}\n")
            pages = page_name_commu[loc]
            for page in pages:
                # Append each page title to the file followed by a newline character
                file.write(str(page) + "\n")

    print(f"[FINISH] Page titles successfully exported to: {community_results_dir}")
    return

In [ ]:
import os
import pickle
import pandas as pd
from tqdm.auto import tqdm

def Community_information(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Generates summary statistics for each identified community, including total edits,
    unique pages, and active editors. The consolidated data is structured into a 
    Pandas DataFrame for comparative analysis and visualization.
    """
    
    # DATA_DIR: Source directory for partitioned network data and history logs
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset')
    
    def load_files():
        """
        Retrieves serialized partition results and filtered editor activity histories.
        """
        # Construct absolute file paths using standardized naming conventions
        partition_file_path = os.path.join(DATA_DIR, f'leiden_single_partition_{page_edit_number}_{editor_edit_number}.pickle')
        with open(partition_file_path, 'rb') as fr:
            partition = pickle.load(fr)
            
        history_file_path = os.path.join(DATA_DIR, 'conditioned_history', f'condi_pages_{page_edit_number}_editors_{editor_edit_number}.pickle')
        with open(history_file_path, 'rb') as fr:
            editor_histories = pickle.load(fr)
            
        return partition, editor_histories
    partition, editor_histories = load_files()
    
    NUM_COMMUNITIES = 19
    # Thematic keywords representing the topical focus of each community cluster
    Keywords = ["Entertainment",           # 0
                "Geopolitics",             # 1
                "US Politics & Society",   # 2
                "Music",                   # 3
                "Science & Nature",        # 4
                "UK & Commonwealth",       # 5
                "South Asian Culture",     # 6
                "Association Football",    # 7
                "Technology & Gaming",     # 8
                "Humanities",              # 9
                "North American Sports",   # 10
                "Combat Sports",           # 11
                "Automotive",              # 12
                "Aviation",                # 13
                "Philippines & Telenovelas", # 14
                "Tennis",                  # 15
                "Chronology",              # 16
                "COVID-19 Pandemic",       # 17
                "Meteorology"              # 18
            ]
    
    # Partition-wise metric counters
    page_partition = {}
    edit_partition = {}
    editor_partition = {}
    for loc in range(19):
        page_partition[loc] = 0
        edit_partition[loc] = 0
        editor_partition[loc] = 0
    
    # Calculate total unique pages per community
    for page in tqdm(partition.keys()):
        loc = partition[page]
        page_partition[loc] += 1
    
    # Aggregate edit counts and distinct editor participation per community
    for editor in tqdm(editor_histories.keys()):
        loc_list = []
        events = editor_histories[editor]
        for timestamp, page in events:
            loc = partition[page]
            edit_partition[loc] += 1
            loc_list.append(loc)
        # Ensure an editor is counted only once per community based on their participation
        loc_list = list(set(loc_list))
        for loc in loc_list:
            editor_partition[loc] += 1
    # Graphical markers assigned for consistent visualization across various plots
    for_markers = [
        'Blue $\triangle$', 'Light Blue $\circ$', 'Orange $\square$', 'Light Orange $\star$', 'Green $\times$', 'Light Green $\pentagon$', 'Red $+$', 'Light Red $\triangleright$', 'Purple $\lozenge$', 'Light Purple $\triangle$', 'Brown $\cir$', 'Light Brown $\squre$', 'Pink $\star$', 'Light Pink $\times$', 'Gray $\pentagon$','Light Gray $+$', 'Olive $\triangleright$', 'Light Olive $\lozenge$', 'Cyan $\vartriangle$'
                  ]
    
    # Consolidate metrics into a structured repository
    data_to_add = []
    df = pd.DataFrame(columns=["Colored marker", "keyword", "edits", "pages", "editors"])

    for loc in range(19):
        new_row_dict = {
            "Colored marker": for_markers[loc],
            "keyword": Keywords[loc],
            "edits": edit_partition[loc],
            "pages": page_partition[loc],
            "editors": editor_partition[loc]
        }
        data_to_add.append(new_row_dict)

    # Construct the final DataFrame for analytical reporting
    df = pd.DataFrame(data_to_add)
    print(df)
    
    # Serialize the summary statistics to the standardized target path
    summary_file_path = os.path.join(DATA_DIR, f'Community_information_{page_edit_number}_{editor_edit_number}.pickle')
    with open(summary_file_path, 'wb') as fw:
        pickle.dump(df, fw)
        
    return df

## Step 3-2: Editor type

### Step 3-2-1: Extract Bots

In [ ]:
import os
import time
import pickle
import requests
import re
import html
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# Configuration and Global Constants
API_URL = "https://en.wikipedia.org/w/api.php"
INFO_PAGE_BASE_URL = "https://en.wikipedia.org/w/index.php"
CATEGORY = "Category:All_Wikipedia_bots"

# Standardized User-Agent to comply with Wikimedia API policies
HEADERS = {
    'User-Agent': 'MyWikipediaBotScraper (yourname@example.com) - Fetching User IDs via action=info'
}

# Define standardized directory paths for data management
DATA_DIR = os.path.join(BASE_DIR, 'Dataset')
os.makedirs(DATA_DIR, exist_ok=True)

# Bot Name Extraction via MediaWiki API
def get_all_category_members(category_title):
    """
    Retrieves all members within a specified MediaWiki category using pagination.
    This function manages continuation tokens to ensure exhaustive data retrieval.
    """
    
    all_bot_names = set()
    cmcontinue = None # Token for the next batch of results
    
    print(f"--- Stage 1: Extracting members from '{category_title}' ---")
    while True:
        # Define API parameters for category member query
        params = {
            "action": "query",
            "format": "json",
            "list": "categorymembers",
            "cmtitle": category_title, 
            "cmlimit": "500",
            "cmprop": "title"
        }
        
        if cmcontinue:
            params['cmcontinue'] = cmcontinue

        try:
            # Implementing a polite delay to comply with server load policies
            time.sleep(0.5) 
            
            response = requests.get(API_URL, headers=headers, params=params, timeout=15)
            response.raise_for_status()
            data = response.json()
            
            # Extracting titles and filtering namespace prefixes
            if 'query' in data and 'categorymembers' in data['query']:
                for member in data['query']['categorymembers']:
                    title = member['title']
                    
                   # Normalizing bot names by removing 'User:' or 'Category:' prefixes
                    if title.startswith("User:"):
                        bot_name = title[5:]
                    elif title.startswith("Category:"):
                        # Exclude sub-categories from the list
                        continue 
                    else:
                        bot_name = title
                        
                    all_bot_names.add(bot_name.strip())

            # Managing pagination logic via continuation tokens
            if 'continue' in data and 'cmcontinue' in data['continue']:
                cmcontinue = data['continue']['cmcontinue']
                print(f"[LOG] {len(all_bot_names)} members retrieved. Proceeding to next token...")
            else:
                print("[SUCCESS] Comprehensive category extraction complete.")
                break

        except requests.exceptions.RequestException as e:
            print(f"[ERROR] API request failed: {e}")
            break
            
    return sorted(list(all_bot_names))

# Execute Stage 1 and serialize the raw name list
bot_list = get_all_category_members(CATEGORY)
bot_list_file_path = os.path.join(DATA_DIR, 'Bot_list.pickle')

with open(bot_list_file_path, 'wb') as fw:
    pickle.dump(bot_list, fw)

# User ID Parsing via HTML Information Pages

def get_userids_from_info_page(bot_name_list):
    """
    Parses the 'action=info' HTML interface for each bot to extract unique User IDs.
    Locates numeric identifiers by targeting the 'mw-pageinfo-user-id' attribute.
    """
    
    user_id_map = {} # Mapping: {bot_name: user_id_str}
    print(f"\n--- Stage 2: Parsing User IDs for {len(bot_name_list)} bots ---")

    for name in tqdm(bot_name_list):
        # Construct the specific URL for the user information interface
        url = f"{INFO_PAGE_BASE_URL}?title=User:{name}&action=info"
        
        try:
            # Short delay to mitigate high-frequency request overhead
            time.sleep(0.1) 
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Targeting the metadata row identified by the unique User ID tag
            user_id_row = soup.find('tr', {'id': 'mw-pageinfo-user-id'})
            
            if user_id_row:
                # The numeric ID is located within the second table data (td) element
                td_elements = user_id_row.find_all('td')
                
                if len(td_elements) >= 2:
                    user_id_value = td_elements[1].text.strip()
                    
                    # Validate numeric integrity before mapping
                    if user_id_value.isdigit():
                        user_id_map[name] = user_id_value

        except (requests.exceptions.RequestException, Exception):
            # Gracefully bypass connection errors or parsing anomalies
            continue

    print("[SUCCESS] User ID parsing complete.")
    return user_id_map

# Data Integration and Serialization

user_ids_by_name = get_userids_from_info_page(bot_list)

final_bot_data = []
for name in bot_list:
    user_id = user_ids_by_name.get(name) 
    if user_id: 
        final_bot_data.append({
            'id': user_id,
            'name': name
        })

# Final validation and data serialization
if final_bot_data:
    final_bot_data.sort(key=lambda x: x['name'])

    print(f"\n[INFO] Total bots integrated with unique IDs: {len(final_bot_data)}")
    
    # Define absolute file path for the integrated bot dataset
    integrated_bot_file_path = os.path.join(DATA_DIR, 'Bot_list_with_info_id.pickle')
    
    try:
        with open(integrated_bot_file_path, 'wb') as fw:
            pickle.dump(final_bot_data, fw)
        print(f"✅ [FINISH] Integrated dataset serialized to: '{integrated_bot_file_path}'")
    except Exception as e:
        print(f"❌ [ERROR] Serialization failed: {e}")
else:
    print("\n[CRITICAL] No matching bot data integrated. Verify HTML parsing patterns.")
    
# This page was last edited on 8 June 2025, at 23:14 (UTC).
# Access 28 October 2025, at 4 pm (KST)

### Step 3-2-2: Calculate Entropy and inverse_lambda

In [ ]:
import os
import pickle
import numpy as np
from tqdm.auto import tqdm

def Calculate_entropy_and_inverse_lambda(page_edit_number, editor_edit_number):
    """
    Calculates structural diversity metrics, specifically Shannon Entropy and the Inversed Simpson Index (inverse_lambda),
    to quantify the distribution of editor activities across identified communities.
    """
    
    # DATA_DIR: Source directory for partitioned network data and activity histories
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset')
    
    def load_files():
        """
        Retrieves serialized community partitions, filtered editor histories, and bot identifiers.
        """
        # Constructing absolute file paths using standardized naming conventions
        partition_file_path = os.path.join(DATA_DIR, f"leiden_partition_{page_edit_number}_{editor_edit_number}.pickle")
        with open(partition_file_path, "rb") as f:
            node_to_partition = pickle.load(f)

        history_file_path = os.path.join(DATA_DIR, 'conditioned_history', f'condi_pages_{page_edit_number}_editors_{editor_edit_number}.pickle')
        with open(history_file_path, 'rb') as fr:
            editor_history = pickle.load(fr)

        bot_list_file_path = os.path.join(DATA_DIR, 'Bot_list_with_info_id.pickle')
        with open(bot_list_file_path, 'rb') as fr:
            Bot_id_name = pickle.load(fr)
            
        return node_to_partition, editor_history, Bot_id_name
    
    # Initializing datasets for metric computation
    node_to_partition, editor_history, Bot_id_name = load_files()
    print('[INFO] Input datasets successfully loaded.')
    
    def get_entropy_inverse_lambda(page_commu, set_pages, C_i):
        """
        Computes entropy and the Inversed Simpson Index for an individual editor.
        The distribution is normalized by C_i (total community activity) to adjust for varying community sizes.
        """
        total_edit = sum(page_commu.values())
        denominator = []
        editor_edit_data = []
        
        # Calculate the normalization factor across associated communities
        for community in page_commu.keys():
            try:
                edit_per_commu = page_commu[community]
                denominator.append(edit_per_commu/C_i[community])
                editor_edit_data.append(edit_per_commu)
            except KeyError:
                editor_edit_data.append(0)
                pass
        denominator = sum(denominator)

        pclogpc = []
        pci_2 = []
        try_commu = 0
        
        # Compute individual probability components for Shannon entropy and Simpson index (lambda)
        for community in page_commu.keys():
            edit_per_commu = page_commu[community]
            # pc represents the normalized probability (relative contribution) within a community
            pc = (edit_per_commu/C_i[community])/denominator
            pclogpc.append(pc*np.log(pc))
            pci_2.append(pc**2)
            try_commu += 1
            
        entropy = - sum(pclogpc)
        lambda_val = sum(pci_2) # Represents the Simpson Index (λ)
        inverse_lambda = 1 / lambda_val if lambda_val != 0 else 0
        return (entropy, lambda_val, inverse_lambda, total_edit, set_pages, try_commu)
    
    # Aggregate activity distributions per editor
    editors = list(editor_history.keys())
    C_i = {} # Global counter for community-wise cumulative edits
    editor_value = {} # Repository for pre-calculated editor distributions
    
    for editor in tqdm(editors):
        pages = []
        events = editor_history[editor]
        for event in events:
            pages.append(event[1])
            
        set_pages = len(list(set(pages)))
        page_commu = {}
        
        for page in pages:
            community = node_to_partition[page]
            C_i[community] = C_i.get(community, 0) + 1
            page_commu[community] = page_commu.get(community, 0) + 1
        editor_value[editor] = page_commu, set_pages
    
    # Execute structural diversity analysis
    Feature = {}
    for editor in tqdm(editors):
        page_commu, set_pages = editor_value[editor]
        Feature[editor] = get_entropy_inverse_lambda(page_commu, set_pages, C_i)

    # Serialize the finalized feature set containing Inversed Simpson Index results
    output_feature_file_path = os.path.join(DATA_DIR, f'Entropy_inverse_lambda_{page_edit_number}_{editor_edit_number}.pickle')
    with open(output_feature_file_path, 'wb') as fw:
        pickle.dump(Feature, fw)
    return

## Step 3-3: Jaccard similarity

In [ ]:
import os
import pickle
import math
import ciso8601
from datetime import datetime, timedelta
from tqdm.auto import tqdm

def Calculate_Jaccard(BASE_DIR, page_edit_number, editor_edit_number, Deltat, part):
    """
    Computes the Jaccard similarity index between editor navigation trajectories and the underlying 
    Wikipedia hyperlink network to quantify the structural alignment of user behavior.
    """
    
    # DATA_DIR: Source directory for partitioned datasets and network topology
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset')
    
    def load_files():
        """
        Retrieves serialized editor histories and the global hyperlink adjacency list.
        """
        # Constructing absolute file paths using standardized naming conventions
        history_file_path = os.path.join(DATA_DIR, 'conditioned_history', f'condi_pages_{page_edit_number}_editors_{editor_edit_number}.pickle')
        with open(history_file_path, 'rb') as fr:
            editor_history = pickle.load(fr)
            
        pair_list_file_path = os.path.join(DATA_DIR, 'conditioned_history', f'condi_pairs_{page_edit_number}_{editor_edit_number}.pickle')
        with open(pair_list_file_path, 'rb') as fr:
            pair_list = pickle.load(fr)
        
        return editor_history, pair_list
    
    editor_history, pair_list = load_files()
    
    # Constructing an adjacency list for efficient structural neighbor lookups
    adj = {}
    for p1, p2 in pair_list:
        if p1 not in adj:
            adj[p1] = set()
        adj[p1].add(p2)
    
    del pair_list # Memory optimization: release the raw edge list
    
    editors = list(editor_history.keys())
    total_length = len(editors)
    num_parts = 40
    
    if part == 1:
        start_index, end_index = math.floor(total_length * 0), math.floor(total_length * (1 / num_parts))
    elif part == 2:
        start_index, end_index = math.floor(total_length * (1 / num_parts)), math.floor(total_length * (5 / num_parts))
    elif part == 3:
        start_index, end_index = math.floor(total_length * (5 / num_parts)), math.floor(total_length * (10 / num_parts))
    elif part == 4:
        start_index, end_index = math.floor(total_length * (10 / num_parts)), total_length
    part_range = [start_index, end_index]
        
    results_dir = {}
    results_undir = {}
    
    # Optimization: local variable assignment for high-speed datetime parsing via ciso8601
    parse_datetime = ciso8601.parse_datetime

    for neditor in tqdm(range(part_range[0], part_range[1])):
        editor = editors[neditor]
        hist = editor_history[editor]
        
        if len(hist) < 2:
            results_dir[editor] = results_undir[editor] = 'NaN'
            continue
            
        # Chronological sorting of revision events
        hist.sort(key=lambda x: x[0]) 

        trajectory_set = set()
        visited_pages = set()
        
        # Iterating through consecutive revisions to identify temporal navigation edges
        for i in range(len(hist) - 1):
            t1_str, p1 = hist[i]
            t2_str, p2 = hist[i+1]
            visited_pages.add(p1)
            visited_pages.add(p2)
            
            # Utilizing ciso8601 for high-performance timestamp comparison (faster than strptime)
            t1 = parse_datetime(t1_str)
            t2 = parse_datetime(t2_str)
            
            # Record edges where the time interval between edits is within the Deltat threshold
            if (t2 - t1).total_seconds() <= Deltat:
                trajectory_set.add((p1, p2))

        if not trajectory_set:
            results_dir[editor] = results_undir[editor] = 'NaN'
            continue

        # Extraction of localized hyperlinks based on the editor's visited page subspace
        hyperlink_set = set()
        for p1 in visited_pages:
            if p1 in adj:
                # Intersect adjacency neighbors with visited pages to isolate relevant network edges
                targets = adj[p1] & visited_pages
                for p2 in targets:
                    hyperlink_set.add((p1, p2))

        # Calculation of Directed Jaccard Index
        inter_dir = len(trajectory_set & hyperlink_set)
        union_dir = len(trajectory_set | hyperlink_set)
        results_dir[editor] = inter_dir / union_dir if union_dir > 0 else 0

        # Calculation of Undirected Jaccard Index (Symmetric connectivity analysis)
        # Optimized tuple creation by conditional ordering to avoid 'sorted()' overhead
        wod_traj_set = { (p[0], p[1]) if p[0] <= p[1] else (p[1], p[0]) for p in trajectory_set }
        wod_hyper_set = { (p[0], p[1]) if p[0] <= p[1] else (p[1], p[0]) for p in hyperlink_set }
        
        inter_undir = len(wod_traj_set & wod_hyper_set)
        union_undir = len(wod_traj_set | wod_hyper_set)
        results_undir[editor] = inter_undir / union_undir if union_undir > 0 else 0
    
    # Consolidate directed and undirected results for serialization
    RESULTS = {'direction': results_dir, 'undirection': results_undir}
        
    # Serialize the finalized Jaccard metrics to the standardized output path
    jaccard_file_path = os.path.join(DATA_DIR, 'Jaccard', f'Jaccard_{page_edit_number}_{editor_edit_number}_deltat_{Deltat}_part{part}.pickle')
    with open(jaccard_file_path, 'wb') as fw:
        pickle.dump(RESULTS, fw)
        
    return

In [ ]:
import os
import pickle

def Merge_Jaccard(BASE_DIR, page_edit_number, editor_edit_number, Deltat):
    """
    Consolidates partitioned Jaccard similarity results into a unified dataset.
    This function merges directed and undirected Jaccard indices across multiple 
    computation segments to produce a comprehensive behavioral alignment profile.
    """
    
    # DATA_DIR: Source directory for partitioned Jaccard metric files
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset', 'Jaccard')
    
    # Initializing repositories for consolidated directed and undirected metrics
    Jaccard = {}
    wod_Jaccard = {}
    
    # Iterate through the pre-defined partitions (Parts 1 to 4) to aggregate data
    for part in range(1, 5):
        # Construct the absolute file path for each partitioned result segment
        part_file_path = os.path.join(DATA_DIR, f'Jaccard_{page_edit_number}_{editor_edit_number}_deltat_{Deltat}_part{part}.pickle')
        
        with open(part_file_path, 'rb') as fr:
            part_Jaccard = pickle.load(fr)
        
        # Extract and update mappings for directed and undirected structural similarity
        j_dict = part_Jaccard['direction']
        J_dict = part_Jaccard['direction']
        wod_J_dict = part_Jaccard['undirection']
        Jaccard.update(J_dict)
        wod_Jaccard.update(wod_J_dict)

    # Consolidate aggregated dictionaries into a final result structure
    RESULTS = {
        'direction': Jaccard, 
        'undirection': wod_Jaccard
    }
        
    # Define the standardized output path for the finalized integrated dataset
    integrated_jaccard_file_path = os.path.join(DATA_DIR, f'Jaccard_{page_edit_number}_{editor_edit_number}_deltat_{Deltat}.pickle')
    
    # Serialize the comprehensive Jaccard metrics to the target directory
    with open(integrated_jaccard_file_path, 'wb') as fw:
        pickle.dump(RESULTS, fw)
        
    print(f"[FINISH] Jaccard data consolidation complete. Saved to: {integrated_jaccard_file_path}")
    return

## Step 3-4: Specific bot

In [ ]:
import os
import pickle

# DATA_DIR: Source directory for integrated datasets and metadata
DATA_DIR = os.path.join(BASE_DIR, 'Dataset')

# --- Load Jaccard Similarity Metrics ---
# Constructing the absolute path for the integrated Jaccard dataset
jaccard_file_path = os.path.join(DATA_DIR, 'Jaccard', f'Jaccard_{page_edit_number}_{editor_edit_number}.pickle')
with open(jaccard_file_path, 'rb') as fr:
    j_dict_raw = pickle.load(fr)
# Utilizing undirected Jaccard indices for symmetrical behavior analysis
J_dict = j_dict_raw['undirection']

# --- Load Structural Diversity Metrics (Entropy & Inversed Simpson Index) ---
entropy_file_path = os.path.join(DATA_DIR, f'Entropy_inverse_lambda_{page_edit_number}_{editor_edit_number}.pickle')
with open(entropy_file_path, 'rb') as fr:
    Entropy_dict = pickle.load(fr)

# --- Load Filtered Editor Histories ---
history_file_path = os.path.join(DATA_DIR, 'conditioned_history', 'condi_pages_1000_editors_1000.pickle')
with open(history_file_path, 'rb') as fr:
    conditioned_history = pickle.load(fr)

# --- Load Page Metadata (ID-to-Title Mapping) ---
page_id_file_path = os.path.join(DATA_DIR, 'raw_page2page', 'page_id_dict.pickle')
with open(page_id_file_path, 'rb') as fr:
    page_id_dict = pickle.load(fr)

# --- Load and Pre-process Bot Identifier Metadata ---
bot_list_file_path = os.path.join(DATA_DIR, 'Bot_list_with_info_id.pickle')
with open(bot_list_file_path, 'rb') as fr:
    Bot_list_with_info_id = pickle.load(fr)

# Constructing a lookup table for bot identifiers (ID_Name to Name)
Bot_ID = {}
for stuff in Bot_list_with_info_id:
    # Mapping standardized identifier strings to bot names
    Bot_ID[f"{stuff['id']}_{stuff['name']}"] = stuff['name']

# Quantitative Verification: Displaying metrics for a specific target editor (PseudoBot)
print(f"Entropy Metrics (PseudoBot): {Entropy_dict.get('6290400_PseudoBot')}")
print(f"Jaccard Index (PseudoBot): {J_dict.get('6290400_PseudoBot')}")

# --- Qualitative Analysis of Target Editor Trajectory ---
# Reconstructing the edit sequence to identify specific content patterns
histories = conditioned_history['6290400_PseudoBot']
for event in histories:
    # Converting page identifiers to canonical Wikipedia titles
    page_title = str(page_id_dict[int(event[1])][0])
    
    # Parsing the title string to detect chronological or system-level entities
    try:
        content_part = page_title.split("'")[1]
        primary_token = content_part.split("_")[0]
        
        # Filter: Exclude standard chronological entities (Month/Year articles)
        months = [
            'January', 'February', 'March', 'April', 'May', 'June', 
            'July', 'August', 'September', 'October', 'November', 'December'
        ]
        
        if primary_token in months:
            continue
        else:
            # Output unique entities that are neither chronological nor strictly numeric
            if not content_part.isdigit():
                print(f"[TARGET ENTITY] {page_id_dict[int(event[1])]}")
    except IndexError:
        # Gracefully handle non-standard title formats
        continue